<a href="https://colab.research.google.com/github/onizuka465/Treinamento-Transformers/blob/main/lab05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers torch -q

In [ ]:
from datasets import load_dataset
import torch
import torch.nn as nn
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


dataset = load_dataset("bentrevett/multi30k")
subset = dataset["train"].select(range(1000))

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

In [ ]:
print (subset[0])

In [ ]:
def tokenizar_par(exemplo):
    en = tokenizer.encode(exemplo["en"], add_special_tokens=False)
    de = [tokenizer.cls_token_id] + tokenizer.encode(exemplo["de"], add_special_tokens=False) + [tokenizer.sep_token_id]
    return {"en_ids": en, "de_ids": de}

subset_tokenizado = [tokenizar_par(subset[i]) for i in range(len(subset))]

print(subset_tokenizado[0])

In [ ]:
def padding(sequencias, pad_id=0):
    max_len = max(len(s) for s in sequencias)
    return [s + [pad_id] * (max_len - len(s)) for s in sequencias]

en_ids = padding([s["en_ids"] for s in subset_tokenizado])
de_ids = padding([s["de_ids"] for s in subset_tokenizado])

print("Shape EN:", len(en_ids), "x", len(en_ids[0]))
print("Shape DE:", len(de_ids), "x", len(de_ids[0]))

In [ ]:
en_tensor = torch.tensor(en_ids, dtype=torch.long)
de_tensor = torch.tensor(de_ids, dtype=torch.long)

print("Shape EN:", en_tensor.shape)
print("Shape DE:", de_tensor.shape)

Modelo do Lab 4 (Adaptado pra biblioteca do Pytorch)

In [ ]:

class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, d_ff=512, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.WQ = nn.Linear(d_model, d_model, bias=False)
        self.WK = nn.Linear(d_model, d_model, bias=False)
        self.WV = nn.Linear(d_model, d_model, bias=False)
        self.W1 = nn.Linear(d_model, d_ff)
        self.W2 = nn.Linear(d_ff, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size)
        self.n_layers = n_layers

    def forward(self, src, tgt):
        X = self.embedding(src)
        y = self.embedding(tgt)

        for _ in range(self.n_layers):
            X = self.encoder_block(X)

        for _ in range(self.n_layers):
            y = self.decoder_block(y, X)

        return self.out(y)

    def encoder_block(self, X):
        X_att = self.self_attention(X, X, X)
        X_norm1 = self.norm(X + X_att)
        X_ffn = torch.relu(self.W1(X_norm1))
        X_ffn = self.W2(X_ffn)
        return self.norm(X_norm1 + X_ffn)

    def decoder_block(self, y, Z):
        y_att = self.self_attention(y, y, y, causal=True)
        y_norm1 = self.norm(y + y_att)
        y_cross = self.self_attention(y_norm1, Z, Z)
        y_norm2 = self.norm(y_norm1 + y_cross)
        y_ffn = torch.relu(self.W1(y_norm2))
        y_ffn = self.W2(y_ffn)
        return self.norm(y_norm2 + y_ffn)

    def self_attention(self, q, k, v, causal=False):
        Q = self.WQ(q)
        K = self.WK(k)
        V = self.WV(v)
        scores = Q @ K.transpose(-2, -1) / (Q.shape[-1] ** 0.5)
        if causal:
            seq_len = Q.shape[1]
            mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool().to(q.device)
            scores = scores.masked_fill(mask, float('-inf'))
        weights = torch.softmax(scores, dim=-1)
        return weights @ V

In [ ]:
vocab_size = tokenizer.vocab_size

modelo = Transformer(vocab_size=vocab_size, d_model=128, d_ff=512, n_layers=2)
criterio = nn.CrossEntropyLoss(ignore_index=0)
otimizador = torch.optim.Adam(modelo.parameters(), lr=0.0001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = modelo.to(device)
en_tensor = en_tensor.to(device)
de_tensor = de_tensor.to(device)

print ("Vocabulário:", vocab_size)

In [ ]:
batch_size = 32
epochs = 20

for epoch in range(epochs):
    total_loss = 0

    for i in range(0, 1000, batch_size):
        src = en_tensor[i:i+batch_size]
        tgt = de_tensor[i:i+batch_size]

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        otimizador.zero_grad()
        saida = modelo(src, tgt_input)
        saida = saida.reshape(-1, vocab_size)
        tgt_output = tgt_output.reshape(-1)

        loss = criterio(saida, tgt_output)
        loss.backward()
        otimizador.step()

        total_loss += loss.item()

    print(f"Época {epoch+1}/{epochs} - Loss: {total_loss/32:.4f}")

Frase que sera utilizada e sua tradução esperada:

Frase: Two young, White males are outside near many bushes. (Primeira frase do subset ingles)

Tradução: Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche. (Primeira frase do subset Alemão)

In [ ]:
modelo.eval()

src = en_tensor[0].unsqueeze(0)

decoder_input = torch.tensor([[tokenizer.cls_token_id]]).to(device)

max_tokens = 50
resultado = []

with torch.no_grad():
    Z = src

    for _ in range(max_tokens):
        saida = modelo(Z, decoder_input)
        next_token = saida[0, -1, :].argmax().item()

        if next_token == tokenizer.sep_token_id:  # <EOS>
            break

        resultado.append(next_token)
        decoder_input = torch.cat([
            decoder_input,
            torch.tensor([[next_token]]).to(device)
        ], dim=1)

print("Tradução gerada:", tokenizer.decode(resultado))
print("Tradução esperada:", subset[0]["de"])